# Trained versus hand-coded executors

Review the saved comparison without retraining. These are fresh semantic-token
pilot results, not recovered weights from the archived paper runs. Local
transition accuracy, composition fidelity, and gradient agreement are separate
measurements. The mixed-format condition was trained on both direct-answer and
trace targets; it is not a corrupted-trace reliability condition.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'experiments/compare_executor_rules.py').exists())
RUN = ROOT / 'results/executor_comparison/pilot_seed42'
manifest = json.loads((RUN / 'manifest.json').read_text())
metrics = pd.read_csv(RUN / 'metrics.csv')
display(Markdown((RUN / 'report.md').read_text()))


## Local transitions

Rows are source states; columns are successor states. The shared color scale
retains probability assigned outside the state vocabulary. Queries use valid
gold prefixes. Outcome-only local tables are omitted because those queries
would be out of distribution.


In [ ]:
display(Image(filename=str(RUN / 'transition_tables.png')))


## Whole-circuit behavior and training

The circuit is chosen in advance and evaluated at all 16 initial states. The
process model generates its trace; direct-answer models condition on COLON.
The main held-out answer accuracy always uses free generation, including the
format decision. The hand-coded and learned architectures differ in capacity.


In [ ]:
display(Image(filename=str(RUN / 'circuit_comparison.png')))
display(Image(filename=str(RUN / 'training_diagnostics.png')))


## Gradient comparison

This measurement uses only the mixed-format model, so both query families have
training support. The actual direct-answer gradient sees no gold trace or answer
in the input. The surrogate differentiates through the Transformer readout and
the product of first-step local tables. A shared-looking heatmap does not imply
that the surrogate predicts the actual gradient.


In [ ]:
display(Image(filename=str(RUN / 'gradient_agreement.png')))
display(metrics[['mode', 'step', 'free_answer_accuracy', 'local_rule_accuracy',
                 'composition_tv_with_invalid', 'gradient_cosine', 'gradient_relative_error']])


## Reuse with models trained in another notebook

See `docs/executor_comparison.md` for `compare_models(...)`, which analyzes live
models without updating their weights. The CLI checkpoints include model and
optimizer states. Run additional seeds before treating the pilot as a robust
main-paper result.
